# Delta Lake — Advanced
Time travel, MERGE (upsert), DELETE e DESCRIBE HISTORY.

> Execute `01_delta_lake_basics.ipynb` antes deste notebook.

In [ ]:
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = configure_spark_with_delta_pip(
    SparkSession.builder
    .master("local[*]")
    .appName("delta-advanced")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
).getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

DELTA_PATH = "/home/jovyan/work/delta-tables"
SILVER     = f"{DELTA_PATH}/silver/transaction"
print("Spark pronto")

## 1. DESCRIBE HISTORY — auditoria completa da tabela

In [ ]:
dt = DeltaTable.forPath(spark, SILVER)
dt.history().select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

## 2. Time Travel — ler versão anterior

In [ ]:
# Versão 0 = escrita inicial (antes do UPDATE de UK → United Kingdom)
v0 = spark.read.format("delta").option("versionAsOf", 0).load(SILVER)
print("Versão 0 — países distintos:")
v0.select("customer_country").distinct().show()

# Versão atual
v_current = spark.read.format("delta").load(SILVER)
print("Versão atual — países distintos:")
v_current.select("customer_country").distinct().show()

## 3. MERGE (upsert) — inserir novos registros ou atualizar existentes

In [ ]:
# Simular chegada de novos dados: 1 update + 1 novo registro
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType

schema = v_current.schema

updates = spark.createDataFrame(
    [
        # Atualizar transaction_id=1001: novo preço
        (1001, 501, "2023-07-11", 101, "Product A", 1, 99.99, 8.23, "john.doe@example.com",
         "+1-234-567-8901", "USA", "New York", 108.22, "John Doe"),
        # Novo registro
        (9999, 601, "2023-08-01", 105, "Product E", 2, 50.00, 5.00, "new.user@example.com",
         "+55-11-99999-9999", "Brazil", "São Paulo", 55.00, "New User"),
    ],
    schema=schema
)

(
    dt.alias("target")
    .merge(updates.alias("source"), "target.transaction_id = source.transaction_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("MERGE executado")
spark.read.format("delta").load(SILVER).filter(F.col("transaction_id").isin(1001, 9999)).show()

## 4. DELETE — remover registros

In [ ]:
before = spark.read.format("delta").load(SILVER).count()

dt.delete(condition=F.col("transaction_id") == 9999)

after = spark.read.format("delta").load(SILVER).count()
print(f"Antes: {before} | Depois: {after} | Removidos: {before - after}")

## 5. Histórico completo após todas as operações

In [ ]:
dt.history().select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

## 6. Restaurar para versão anterior (RESTORE)

In [ ]:
# Descomente para restaurar a versão 0
# dt.restoreToVersion(0)
# spark.read.format("delta").load(SILVER).count()